<p> <center> <a href="../../start_here_ja.ipynb">Home Page</a> </center> </p>

<div>
    <span style="float: left; width: 33%; text-align: left;"><a href="../06_challenge_ja.ipynb">Previous Notebook</a></span>
    <span style="float: left; width: 34%; text-align: center;">
        <a href="../01_inference_endpoint_ja.ipynb">1</a>
        <a href="../02_introduction_mcp_ja.ipynb">2</a>
        <a href="../03_low_level_mcp_ja.ipynb">3</a>
        <a href="../04_langraph_agent_ja.ipynb">4</a>
        <a href="../05_nemo_agent_toolkit_ja.ipynb">5</a>
        <a href="../06_challenge_ja.ipynb">6</a>
        <a >7</a>
    </span>
    <span style="float: left; width: 33%; text-align: right;"></span>
</div>

# OpenClaw

OpenClaw は、質問に答えるだけでなく、実際にアクションを実行することを目指して設計された personal AI アシスタントです — 事実上ほぼすべてのプラットフォームでユーザーの代わりに行動できます。

中核となるのは、自分のデバイス上で動かすセルフホスト型のアシスタントです。WhatsApp、Telegram、Slack、Discord、iMessage、Microsoft Teams、Signal など、すでに使っているコミュニケーションチャネルと統合できるので、どこにいても OpenClaw とやりとりできます。

一般的な AI チャットボットと OpenClaw が異なるのは、常時稼働の能動的な性質です。persistent memory、ペルソナのオンボーディング、コミュニケーション統合、バックグラウンドの heartbeat タスクをサポートしているため、cron ジョブの実行、リマインダーの設定、バックグラウンドでの自律動作が可能です。

OpenClaw はユーザーオーナーシップを重視して構築されています。あなたの context と skill は閉じたエコシステムではなく、あなた自身のコンピューター上で動作します。オープンソースで、新しい skill と機能を継続的に構築する成長中のコミュニティがあります。

テストを自律的に実行して pull request を開く coding コンパニオン、カレンダーや健康データに接続された個人ライフアシスタント、就寝中に email やタスクを処理する能動的な agent — OpenClaw は次世代の personal AI のためのオープンで拡張可能な基盤として位置づけられています。

**この lab では、Nemo Agent Toolkit ベースの workflow を agent skill を通じて OpenClaw agent に統合する手順を解説します。**

**このノートブックの残りでは、新しいターミナルを開いてそこでコマンドを実行してください。**

**OpenClaw はターミナルとファイルシステムへの完全で無制限なアクセスを持つため、隔離された環境(例:[brev.dev](https://brev.nvidia.com/) 上の VM)で実行することを推奨します。**

## セットアップ

以下の手順は `OpenClaw 2026.4.15` を `Ubuntu 22.04.5 LTS (Jammy Jellyfish)` 上でテストしました。
最新の手順は [公式ドキュメント](https://docs.openclaw.ai/start/getting-started) を参照してください。

#### OpenClaw をインストールする

以下のコマンドを実行する

```
curl -fsSL https://openclaw.ai/install.sh | bash
```

`Generate and configure a gateway token` と聞かれたら `yes` を選択

`Create Session store dir at ~/.openclaw/agents/main/sessions` と聞かれたら `yes` を選択

`Enable bash shell completion for openclaw` と聞かれたら `yes` を選択

#### オンボーディングを実行する

以下のコマンドを実行する

```
openclaw onboard --install-daemon
```

`I understand this is personal-by-default and shared/multi-user use requires lock-down.` と聞かれたら `yes` を選択

`Setup mode` では `quickstart` を選択

`Config handling` では `Use existing values` を選択

`Model/auth provider` では `Skip for now` を選択

`Filter models by provider` では `All providers` を選択

`Default model` では `Keep current` を選択

`Select channel` と聞かれたら `Skip for now` を選択

`Search provider` では `Skip for now` を選択

`Configure skills` と聞かれたら `No` を選択

API キーを有効化するか聞かれたら `No` を選択

`Enable hooks` と聞かれたら `Skip for now` を選択

OpenClaw gateway は OpenClaw を常時動作させるバックグラウンドの Node.js プロセスです。
接続されたチャネル(TUI、ダッシュボード、Telegram など)からの受信メッセージを listen し、それを適切な agent にルーティングします。
これを systemd ユーザーサービスとしてインストールしておくと、ログイン時に自動再起動するので、毎回手動で起動しなくても agent に常にアクセス可能になります。

#### OpenClaw gateway サービスを起動する

以下のコマンドを実行する

```
export XDG_RUNTIME_DIR=/run/user/$(id -u)
export DBUS_SESSION_BUS_ADDRESS=unix:path=${XDG_RUNTIME_DIR}/bus
openclaw gateway install
openclaw gateway status
```

想定される出力:

```
Service: systemd (enabled)
File logs: /tmp/openclaw/openclaw-2026-04-21.log
Command: /usr/bin/node /home/ubuntu/.npm-global/lib/node_modules/openclaw/dist/index.js gateway --port 18789
Service file: ~/.config/systemd/user/openclaw-gateway.service
Service env: OPENCLAW_GATEWAY_PORT=18789

Config (cli): ~/.openclaw/openclaw.json
Config (service): ~/.openclaw/openclaw.json

Gateway: bind=loopback (127.0.0.1), port=18789 (service args)
Probe target: ws://127.0.0.1:18789
Dashboard: http://127.0.0.1:18789/
Probe note: Loopback-only gateway; only local clients can connect.

Runtime: running (pid 641834, state active, sub running, last exit 0, reason 0)
RPC probe: ok

Listening: 127.0.0.1:18789
```

## OpenClaw を設定する

OpenClaw は openclaw CLI を使うか、config ファイルを直接編集するかのいずれかで設定できます。ここでは両方のアプローチを使用します。

#### デフォルトの openclaw config をバックアップする。

```
cp $HOME/.openclaw/openclaw.json $HOME/.openclaw/openclaw.json.default
```

#### music store agent を作成する

```
openclaw agents add music-store-agent
```

Workspace ディレクトリを聞かれたら `ENTER` を押す

agent のモデル/auth を設定するか聞かれたら `No` を選択

chat チャネルを設定するか聞かれたら `No` を選択

想定される出力:

```
Updated ~/.openclaw/openclaw.json
Workspace OK: ~/.openclaw/workspace/music-store-agent
Sessions OK: ~/.openclaw/agents/music-store-agent/sessions
```

#### モデル属性を指定する

OpenClaw の LLM バックエンドとして [nemotron 3 super 120b](https://build.nvidia.com/nvidia/nemotron-3-super-120b-a12b) モデルを使用します。
このモデルは最大 1M トークンをサポートしていますが、context window は 100k トークンに制限しています。

以下の json で `<your_api_key>` を自分の API キーに置き換え、
`$HOME/.openclaw/openclaw.json` に書き込みます。

```json
"models": {
    "mode": "merge",
    "providers": {
        "nvidia-nim": {
            "baseUrl": "https://integrate.api.nvidia.com/v1",
            "apiKey": "<your_api_key>",
            "api": "openai-completions",
            "models": [
                {
                    "id": "nvidia/nemotron-3-super-120b-a12b",
                    "name": "nvidia/nemotron-3-super-120b-a12b (Custom Provider)",
                    "reasoning": false,
                    "input": [
                        "text"
                    ],
                    "cost": {
                    "input": 0,
                    "output": 0,
                    "cacheRead": 0,
                    "cacheWrite": 0
                    },
                    "contextWindow": 100000,
                    "maxTokens": 4096
                }
            ]
        }
    }
}
```

#### agent 属性を指定する

`$HOME/.openclaw/openclaw.json` の `music-store-agent` に `model` と `default` のキーを追加する。

```json
"agents": {
    ...
    "list": [
      ...
      {
        "id": "music-store-agent",
        "name": "music-store-agent",
        ...
        "model": "nvidia/nemotron-3-super-120b-a12b",
        "default": true
      }
    ]
  }
```

#### music store agent を定義する

`SOUL.md` で agent の振る舞いを定義する。

`SOUL.md` は agent の identity と振る舞いを定義するファイルです。OpenClaw は起動時にこれを読み込んで agent の system prompt を構築します。`## Allowed Skills` セクションはアクセス制御リストとして機能し、agent はここに列挙された skill のみを呼び出そうとするので、意図しない機能の使用を防ぎます。

```
cat > $HOME/.openclaw/workspace/music-store-agent/SOUL.md << 'EOF'
# Music Store Agent

You are a music store agent. Do not invent anything. All data comes from skills.

## Allowed Skills
- music-store-assistant
EOF
```

#### agent skill を作成する

music store agent の中に skills フォルダーを作成する。

```
mkdir -p $HOME/.openclaw/workspace/music-store-agent/skills/music-store-assistant
```

`SKILL.md` に内容を追加する。

この指示は agent に対し、NAT webserver の endpoint を叩いて workflow を呼び出す `curl` コマンドを発行するよう指示している。

`SKILL.md` は YAML front-matter ブロック(`name`、`description`)で skill 発見に使われ、その後に agent がランタイムで読む自由形式のマークダウン指示が続きます。`description` は agent が*いつ* skill を呼び出すかを判断する助けになり、本体は*どう*呼び出すかを伝えます。重要なのは、agent はこのファイルを動的に読み込むため、gateway を再起動しなくても skill の指示を更新できる点です。

```
cat > $HOME/.openclaw/workspace/music-store-agent/skills/music-store-assistant/SKILL.md << 'EOF'
---
name: music-store-assistant
description: lookup information on tracks,artists,abums and process refunds.
---

### Query the music store assistant

Replace <query> with question from user/agent. Do not remove any information.

```bash
curl -X POST http://localhost:8001/generate \
  -H "Content-Type: application/json" \
  -d $'{
    "input_message": <query>
  }'
EOF
```

#### 上記の変更を反映するため openclaw gateway を再起動する

```
openclaw gateway restart
```

#### workflow endpoint を起動する

agent skill は `http://localhost:8001/generate` を呼び出します。このリクエストを処理するためにバックグラウンドで 2 つのサービスが動いている必要があります:
- **MCP invoice server** (`mcp-server-invoice`) — invoice データベースを Model Context Protocol 経由で呼び出し可能な tool として公開する(顧客検索、invoice 行クエリ等)。
- **NAT web server** — Nemo Agent Toolkit パイプラインをポート 8001 で実行する。自然言語クエリを受け取り、LLM を呼び出し、必要に応じて MCP tool を呼び出し、最終回答を返す。

mcp invoice server を起動する。

```
uv run --directory agentic-ai-bootcamp/challenge/mcp-servers/invoice mcp-server-invoice &
```

nat web server を起動する

```
source agentic-ai-bootcamp/challenge/nemo_agent_toolkit/.venv/bin/activate
nat serve --port 8001 --config_file agentic-ai-bootcamp/challenge/nemo_agent_toolkit/workflow.yaml &
```

#### web endpoint をテストする

```
curl -X POST http://localhost:8001/generate \
  -H "Content-Type: application/json" \
  -d $'{
    "input_message": "my name is Aaron Mitchell and my number is +1 (204) 452-6452. Give me the invoice line IDs related to the artist Led Zeppelin"
  }'
```

想定される出力:

```json
{"value":"The invoice line IDs for tracks by Led Zeppelin are **267** and **268**."}
```

#### OpenClaw TUI を起動する

OpenClaw TUI はテキストベースのターミナル内ダッシュボードで、OpenClaw agent の操作とモニタリングに使えます。

```
openclaw tui
```

あるいは、OpenClaw ダッシュボードを起動して、その URL にブラウザでアクセスすることもできます。

```
openclaw dashboard --no-open
```

#### music store agent にクエリを投げる

openclaw tui に以下を入力してください

```
my name is Aaron Mitchell and my number is +1 (204) 452-6452. Give me the invoice line IDs related to the artist Led Zeppelin
```

想定される出力:

```
The invoice line IDs related to the artist Led Zeppelin for Aaron Mitchell are **267** and **268**.
```

#### セッション trace を確認する

agent の sessions フォルダーのファイル一覧を表示する

```
ls $HOME/.openclaw/agents/music-store-agent/sessions/
```

想定される出力:

```
38cadd64-fd9b-4858-93e5-f3fe45ea71d3.jsonl  sessions.json
```

`<sessionId>` をディレクトリ内で見つけた jsonl ファイル名に置き換える

```
cat $HOME/.openclaw/agents/music-store-agent/sessions/<sessionId>.jsonl
```

想定される出力:

```json
{"role":"user","content":[{"type":"text","text":"Sender (untrusted metadata):\n```json\n{\n  \"label\": \"openclaw-tui\",\n  \"id\": \"openclaw-tui\",\n  \"name\": \"openclaw-tui\",\n  \"username\": \"openclaw-tui\"\n}\n```\n\n[Fri 2026-04-10 07:39 UTC] my name is Aaron Mitchell and my number is +1 (204) 452-6452. Give me the invoice line IDs related to the artist Led Zeppelin"}],"timestamp":1775806742065}}

{"role":"assistant","content":[{"type":"thinking","thinking":"We need to get the invoice line IDs related to artist Led Zeppelin. First, I should check the available skills and see what tools I have. Looking at the skills, there's a music-store-assistant skill. I should read that to understand how to use it. Let me read the SKILL.md for music-store-assistant.\n\n","thinkingSignature":"reasoning_content"},{"type":"toolCall","id":"chatcmpl-tool-983ae2ee723dbbd6","name":"read","arguments":{"path":"/home/ubuntu/.openclaw/workspace-music-store-agent/skills/music-store-assistant/SKILL.md"}}],"api":"openai-completions","provider":"nvidia-nim","model":"nvidia/nemotron-3-super-120b-a12b","usage":{"input":13920,"output":115,"cacheRead":0,"cacheWrite":0,"totalTokens":14035,"cost":{"input":0,"output":0,"cacheRead":0,"cacheWrite":0,"total":0}},"stopReason":"toolUse","timestamp":1775806742067,"responseId":"chatcmpl-965f24e5c9cbb680"}}

{"role":"toolResult","toolCallId":"chatcmpl-tool-983ae2ee723dbbd6","toolName":"read","content":[{"type":"text","text":"---\nname: music-store-assistant\ndescription: lookup information on tracks,artists,abums and process refunds.\n---\n\n### Query the music store assistant\n\nReplace <query> with exact statement from user/agent. Do not remove any information.\n\n```bash\ncurl -X POST http://localhost:8001/generate \\\n  -H \"Content-Type: application/json\" \\\n  -d $'{\n    \"input_message\": <query>\n  }'\n```\n"}],"isError":false,"timestamp":1775806755093}}

{"role":"assistant","content":[{"type":"thinking","thinking":"We need to call the music store assistant skill to get invoice line IDs related to artist Led Zeppelin. According to the skill description, I need to use curl to POST to http://localhost:8001/generate with input_message containing the user's query exactly.\n\nUser's query: \"my name is Aaron Mitchell and my number is +1 (204) 452-6452. Give me the invoice line IDs related to the artist Led Zeppelin\"\n\nI should include that exact statement as input_message. Let me call the tool.\n","thinkingSignature":"reasoning_content"},{"type":"toolCall","id":"chatcmpl-tool-bdef75da29fa3b69","name":"exec","arguments":{"command":"curl -X POST http://localhost:8001/generate -H \"Content-Type: application/json\" -d $'{\"input_message\": \"my name is Aaron Mitchell and my number is +1 (204) 452-6452. Give me the invoice line IDs related to the artist Led Zeppelin\"}'"}}],"api":"openai-completions","provider":"nvidia-nim","model":"nvidia/nemotron-3-super-120b-a12b","usage":{"input":14081,"output":209,"cacheRead":0,"cacheWrite":0,"totalTokens":14290,"cost":{"input":0,"output":0,"cacheRead":0,"cacheWrite":0,"total":0}},"stopReason":"toolUse","timestamp":1775806755096,"responseId":"chatcmpl-a3299bb15469dd4c"}}

{"role":"toolResult","toolCallId":"chatcmpl-tool-bdef75da29fa3b69","toolName":"exec","content":[{"type":"text","text":"% Total    % Received % Xferd  Average Speed   Time    Time     Time  Current\n                                 Dload  Upload   Total   Spent    Left  Speed\n\r  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0\r100   271  100   124  100   147  52698  62473 --:--:-- --:--:-- --:--:--  132k\n{\"detail\":[{\"type\":\"json_invalid\",\"loc\":[\"body\",0],\"msg\":\"JSON decode error\",\"input\":{},\"ctx\":{\"error\":\"Expecting value\"}}]}"}],"details":{"status":"completed","exitCode":0,"durationMs":30,"aggregated":"% Total    % Received % Xferd  Average Speed   Time    Time     Time  Current\n                                 Dload  Upload   Total   Spent    Left  Speed\n\r  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0\r100   271  100   124  100   147  52698  62473 --:--:-- --:--:-- --:--:--  132k\n{\"detail\":[{\"type\":\"json_invalid\",\"loc\":[\"body\",0],\"msg\":\"JSON decode error\",\"input\":{},\"ctx\":{\"error\":\"Expecting value\"}}]}","cwd":"/home/ubuntu/.openclaw/workspace-music-store-agent"},"isError":false,"timestamp":1775806766203}}

{"role":"assistant","content":[{"type":"thinking","thinking":"The user is asking for invoice line IDs related to the artist Led Zeppelin. I need to call the music store assistant skill. The skill description says it can lookup information on tracks, artists, albums and process refunds. The user wants invoice line IDs related to Led Zeppelin. I should use the skill's query format. The skill.md shows how to call it: a curl POST to http://localhost:8001/generate with JSON input_message. I need to include the user's message exactly. However, the previous attempt failed because I didn't properly format the JSON. The input_message should be a string containing the user's request. I need to escape it properly. Let me try again with correct JSON formatting. I'll use the exec tool with proper quoting. The user said: \"my name is Aaron Mitchell and my number is +1 (204) 452-6452. Give me the invoice line IDs related to the artist Led Zeppelin\". I'll include that as the input_message. Let's do it.\n","thinkingSignature":"reasoning_content"},{"type":"toolCall","id":"chatcmpl-tool-8fb781cff172d362","name":"exec","arguments":{"command":"curl -X POST http://localhost:8001/generate -H \"Content-Type: application/json\" -d '{\"input_message\": \"my name is Aaron Mitchell and my number is +1 (204) 452-6452. Give me the invoice line IDs related to the artist Led Zeppelin\"}'"}}],"api":"openai-completions","provider":"nvidia-nim","model":"nvidia/nemotron-3-super-120b-a12b","usage":{"input":14372,"output":308,"cacheRead":0,"cacheWrite":0,"totalTokens":14680,"cost":{"input":0,"output":0,"cacheRead":0,"cacheWrite":0,"total":0}},"stopReason":"toolUse","timestamp":1775806766206,"responseId":"chatcmpl-8cd3368d9ce36baa"}}

{"role":"toolResult","toolCallId":"chatcmpl-tool-8fb781cff172d362","toolName":"exec","content":[{"type":"text","text":"% Total    % Received % Xferd  Average Speed   Time    Time     Time  Current\n                                 Dload  Upload   Total   Spent    Left  Speed\n\r  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0\r100   146    0     0  100   146      0    121  0:00:01  0:00:01 --:--:--   121\r100   146    0     0  100   146      0     66  0:00:02  0:00:02 --:--:--    66\r100   146    0     0  100   146      0     45  0:00:03  0:00:03 --:--:--    45\r100   146    0     0  100   146      0     34  0:00:04  0:00:04 --:--:--    34\r 56   257    0     0  100   146      0     29  0:00:05  0:00:04  0:00:01    29\r100   257  100   111  100   146     22     29  0:00:05  0:00:04  0:00:01    29\n{\"value\":\"The invoice line IDs related to the artist Led Zeppelin for Aaron Mitchell are **267** and **268**.\"}"}],"details":{"status":"completed","exitCode":0,"durationMs":4977,"aggregated":"% Total    % Received % Xferd  Average Speed   Time    Time     Time  Current\n                                 Dload  Upload   Total   Spent    Left  Speed\n\r  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0\r100   146    0     0  100   146      0    121  0:00:01  0:00:01 --:--:--   121\r100   146    0     0  100   146      0     66  0:00:02  0:00:02 --:--:--    66\r100   146    0     0  100   146      0     45  0:00:03  0:00:03 --:--:--    45\r100   146    0     0  100   146      0     34  0:00:04  0:00:04 --:--:--    34\r 56   257    0     0  100   146      0     29  0:00:05  0:00:04  0:00:01    29\r100   257  100   111  100   146     22     29  0:00:05  0:00:04  0:00:01    29\n{\"value\":\"The invoice line IDs related to the artist Led Zeppelin for Aaron Mitchell are **267** and **268**.\"}","cwd":"/home/ubuntu/.openclaw/workspace-music-store-agent"},"isError":false,"timestamp":1775806780469}}

{"role":"assistant","content":[{"type":"thinking","thinking":"We have received the response from the music store assistant: the invoice line IDs related to the artist Led Zeppelin for Aaron Mitchell are 267 and 268. I should present this answer to the user. Let's do that.\n","thinkingSignature":"reasoning_content"},{"type":"text","text":"The invoice line IDs related to the artist Led Zeppelin for Aaron Mitchell are **267** and **268**."}],"api":"openai-completions","provider":"nvidia-nim","model":"nvidia/nemotron-3-super-120b-a12b","usage":{"input":14960,"output":77,"cacheRead":0,"cacheWrite":0,"totalTokens":15037,"cost":{"input":0,"output":0,"cacheRead":0,"cacheWrite":0,"total":0}},"stopReason":"stop","timestamp":1775806780473,"responseId":"chatcmpl-9269a85cc94ddd79"}}
```

インタラクションのサマリーを以下に示します。

1. ユーザーリクエスト
Aaron Mitchell(電話番号:+1 (204) 452-6452)が、Led Zeppelin に関連する invoice 行 ID を尋ねるメッセージを送信した。
2. アシスタントの思考
アシスタントは内部で推論し、適切な tool を見つける必要があると判断し、music-store-assistant skill を特定し、まず SKILL.md を読むことにした。
3. アシスタントが skill ファイルを読む
アシスタントは read tool を呼び出して `SKILL.md` を読み、music-store-assistant の使い方を理解した。
4. skill 指示の取得
SKILL.md には、ユーザーのメッセージを `input_message` としてリクエストボディに入れ、`http://localhost:8001/generate` への curl POST リクエストで送信すべきと書かれていた。
5. 1 回目の API 呼び出し(失敗)
アシスタントは curl リクエストを発行したが、`$'...'` という bash の quoting syntax を使ったため JSON パースエラーが発生し、API が JSON decode エラーで応答した。
6. 2 回目の API 呼び出し(成功)
アシスタントは quoting を修正し、標準的なダブルクォートを使って同じリクエストを再送信した。今度は server がリクエストを正常に処理した。
7. API レスポンス受信
music store API は結果を返した:invoice 行 ID 267 と 268 が Aaron Mitchell の Led Zeppelin 購入に紐づいている。
8. ユーザーへの最終回答
アシスタントは結果を明確に伝え、Aaron Mitchell の Led Zeppelin に関する invoice 行 ID が 267 と 268 であると伝えた。

# NemoClaw

NemoClaw は NVIDIA が提供するオープンソーススタックで、自律 AI agent(特に OpenClaw のような常時稼働アシスタント)を運用する際のセキュリティ、プライバシー、信頼性のギャップを解決することを目指しています。
NemoClaw が取り組む問題の概要を以下の表に示します。

| 観点                  | OpenClaw のベースライン                                                                                                         | OpenClaw 上の NemoClaw                                                                                                    |
| ----------------------- | ------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------ |
| システムアクセス           | デフォルトで shell、ファイル、スクリプトへの完全アクセス。広範な権限で動作することが多い                         | サンドボックス化されたランタイム内で動作し、ポリシーで制御された限定的な機能のみを持つ                              |
| ネットワーク露出        | 弱い認証や安全でないデフォルトでインターネットに直接公開されているインスタンスが多い                               | ポリシー経由で出入りのネットワークトラフィックを制御した堅牢なデプロイ向けに設計されている                         |
| Skill エコシステム         | 検証されていないコミュニティ skill が多数あり、マルウェアや有害な指示も報告されている                      | skill 自体を「直す」のではなく、サンドボックスと skill ごとのポリシーで skill ができることを制限する                    |
| プロンプト注入の影響 | 任意のコード実行、認証情報漏洩、データ持ち出しにつながり得る。組み込みの緩和策はほとんどない | プライバシールーターと guardrail で、悪意あるプロンプトが引き起こす機密データフローを検出・ブロック可能          |
| 機密情報とデータ処理 | 平文での鍵漏洩、ログ露出、不明確なデータフロー制御の事例がある                            | サンドボックスの外に出るデータに対する一元的なポリシー。機密アクセスの試行をモニタリング         |
| エンタープライズ制御     | 監査、最小権限、コンプライアンス制御のネイティブサポートは限定的                                    | 監査証跡、構成可能なセキュリティ/プライバシーポリシー、エンタープライズセキュリティツールへの統合を追加 |

NemoClaw の動作についての詳細は [こちら](https://docs.nvidia.com/nemoclaw/latest/about/how-it-works.html) を参照してください。

<p> <center> <a href="../../start_here_ja.ipynb">Home Page</a> </center> </p>

<div>
    <span style="float: left; width: 33%; text-align: left;"><a href="../06_challenge_ja.ipynb">Previous Notebook</a></span>
    <span style="float: left; width: 34%; text-align: center;">
        <a href="../01_inference_endpoint_ja.ipynb">1</a>
        <a href="../02_introduction_mcp_ja.ipynb">2</a>
        <a href="../03_low_level_mcp_ja.ipynb">3</a>
        <a href="../04_langraph_agent_ja.ipynb">4</a>
        <a href="../05_nemo_agent_toolkit_ja.ipynb">5</a>
        <a href="../06_challenge_ja.ipynb">6</a>
        <a >7</a>
    </span>
    <span style="float: left; width: 33%; text-align: right;"></span>
</div>